# Week 09 PEFT — cloud execution
Status: **PARTIAL / GPU BLOCKED** in Codex Cloud. Historical results are retained, not rerun.
Run on Linux cloud CUDA only. Run smoke first, then set SMOKE_ONLY=False and point SMOKE_RESULTS_PATH at its results.json.
The original student notebooks remain unchanged. Dataset correction: canonical codeparrot/codeparrot-clean, each line truncated to 512 characters.


In [ ]:
RUN_LEGACY_EXPERIMENTS = False
RUN_FINAL_EXPERIMENT = True
SMOKE_ONLY = True  # Default: actual model load + two GPU updates + checkpoint reload checks.
SMOKE_RESULTS_PATH = "./python_custom_lora_cloud_smoke/results.json"  # For full runs, set this to the attached successful smoke artifact.
EXPERIMENT_NAME = "python_custom_lora_cloud"  # Use a new name for an intentionally new fresh run.
assert not (RUN_LEGACY_EXPERIMENTS and RUN_FINAL_EXPERIMENT), "Use a separate fresh session for legacy experiments."
HELPER_SOURCE_SHA256 = "48414bb285c33235633a3c1d41c6d474ee95c1ca1be90e78d5a760845eda5e43"


In [ ]:
"""CPU-testable custom LoRA and cloud-only QLoRA experiment utilities.

The notebook embeds this file verbatim; it does not need a repository upload.
No model or dataset is downloaded at import time.
"""

import ast
import hashlib
import json
import math
import os
import random
import time
from copy import deepcopy
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F


@dataclass
class ExperimentConfig:
    base_model: str = "unsloth/Qwen3-8B-Base-bnb-4bit"
    dataset: str = "codeparrot/codeparrot-clean"
    seed: int = 42
    rank: int = 8
    max_characters: int = 512
    sequence_length: int = 512
    train_files: int = 2048
    eval_files: int = 128
    max_scanned: int = 30000
    max_steps: int = 500
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_steps: int = 25
    save_steps: int = 100
    max_new_tokens: int = 128


TARGETS = ("self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj")
PROMPTS = ("", "import", "from", "while", "try", "if", "for", "torch")


class CustomLoRALayer(nn.Module):
    """Original assignment convention: y = base(x) + x A B, without scaling."""

    def __init__(self, module, rank):
        super().__init__()
        if rank < 1:
            raise ValueError("rank must be positive")
        self.module = module
        self.module.requires_grad_(False)
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device, dtype=torch.float32))
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.adapter_A, a=math.sqrt(5))

    def forward(self, inputs):
        base = self.module(inputs)
        # Keep adapter arithmetic in FP32 even inside FP16 autocast (T4).
        context = torch.autocast("cuda", enabled=False) if inputs.is_cuda else nullcontext()
        with context:
            delta = inputs.to(device=self.adapter_A.device, dtype=self.adapter_A.dtype) @ self.adapter_A @ self.adapter_B
        return base + delta.to(device=base.device, dtype=base.dtype)


def seed_everything(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Quantized GPU kernels are not guaranteed bitwise deterministic.
    torch.use_deterministic_algorithms(True, warn_only=True)


def require_cloud_cuda():
    import sys
    cloud_root = any(Path(path).is_dir() for path in ("/kaggle/working", "/content", "/workspace"))
    if sys.platform != "linux" or not cloud_root or not torch.cuda.is_available():
        raise RuntimeError("BLOCKED: Linux cloud CUDA GPU required. Never run training on a local Mac.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise RuntimeError("This notebook uses a single CUDA process; do not launch it with torchrun.")


def memory_snapshot(label):
    require_cloud_cuda()
    result = {"label": label, "devices": []}
    for index in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(index)
        result["devices"].append({"index": index, "name": torch.cuda.get_device_name(index), "free_gib": free / 2**30, "total_gib": total / 2**30, "allocated_gib": torch.cuda.memory_allocated(index) / 2**30, "peak_gib": torch.cuda.max_memory_allocated(index) / 2**30})
    print(json.dumps(result), flush=True)
    return result


def validate_smoke_gate(path, config, metadata):
    """Require completed GPU evidence matching this code, environment and sources.

    This validates a previously retrieved results.json, never creates a pass.
    CPU tests use synthetic evidence explicitly and do not imply GPU execution.
    """
    path = Path(path)
    if not path.is_file():
        raise RuntimeError("Full training blocked: run SMOKE_ONLY=True and attach its successful results.json first")
    evidence = json.loads(path.read_text())
    checks = ("model_load_passed", "gpu_smoke_passed", "checkpoint_reload_logits_passed", "checkpoint_reload_generations_passed")
    if evidence.get("status") != "smoke_only" or not all(evidence.get(key) is True for key in checks):
        raise RuntimeError("Full training blocked: GPU model-load, two-update and reload smoke checks must all pass")
    if evidence.get("training", {}).get("global_step") != 2:
        raise RuntimeError("Full training blocked: smoke must complete exactly two successful optimizer updates")
    previous = evidence.get("metadata", {})
    for key in ("base_model", "model_revision", "dataset", "dataset_revision", "dataset_split", "helper_sha256", "environment", "gpu_name", "execution_device"):
        if key not in metadata or previous.get(key) != metadata[key]:
            raise RuntimeError(f"Full training blocked: smoke metadata mismatch for {key}; rerun smoke on this environment/code")
    if previous.get("execution_device") != "cuda:0":
        raise RuntimeError("Full training blocked: smoke evidence must come from CUDA 0")
    reduced = {"max_steps", "save_steps", "train_files", "eval_files", "max_new_tokens"}
    for key, value in asdict(config).items():
        if key not in reduced and evidence.get("config", {}).get(key) != value:
            raise RuntimeError(f"Full training blocked: smoke hyperparameter mismatch for {key}")
    if evidence.get("gradient_smoke", {}).get("modules_with_finite_nonzero_B_gradients") != 216:
        raise RuntimeError("Full training blocked: incomplete adapter gradient coverage")
    if evidence.get("checkpoint", {}).get("tensors") != 432:
        raise RuntimeError("Full training blocked: incorrect smoke adapter tensor count")
    for stage in ("baseline", "after"):
        value = evidence.get(stage, {}).get("token_nll")
        if not isinstance(value, (int, float)) or not math.isfinite(value):
            raise RuntimeError("Full training blocked: missing finite smoke evaluation metrics")
    return {"path": str(path), "sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "global_step": 2}


def inject_custom_lora(model, rank=8):
    if any(isinstance(module, CustomLoRALayer) for module in model.modules()):
        raise RuntimeError("Adapters are already attached; do not overwrite trained weights.")
    model.requires_grad_(False)
    for layer in model.model.layers:
        for target in TARGETS:
            component, name = target.split(".")
            parent = getattr(layer, component)
            setattr(parent, name, CustomLoRALayer(getattr(parent, name), rank))
    modules = [(name, module) for name, module in model.named_modules() if isinstance(module, CustomLoRALayer)]
    assert len(modules) == 6 * len(model.model.layers)
    trainable = [name for name, value in model.named_parameters() if value.requires_grad]
    assert len(trainable) == 2 * len(modules)
    assert all(name.endswith((".adapter_A", ".adapter_B")) for name in trainable)
    return modules


def custom_adapter_state(model):
    # Also recognizes the original notebook's LoRALayer, without a PEFT flag.
    return {name: value.detach().cpu().clone() for name, value in model.named_parameters() if name.endswith((".adapter_A", ".adapter_B"))}


def save_custom_lora(model, path, metadata):
    state = custom_adapter_state(model)
    if not state or not all(torch.isfinite(value).all().item() for value in state.values()):
        raise ValueError("Cannot save empty or nonfinite adapter weights")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        raise FileExistsError(f"Preserving existing checkpoint: {path}")
    modules = []
    for name, module in model.named_modules():
        if hasattr(module, "adapter_A") and hasattr(module, "adapter_B"):
            modules.append({"name": name, "rank": module.adapter_A.shape[1], "in_features": module.adapter_A.shape[0], "out_features": module.adapter_B.shape[1]})
    payload = {"format": "week09-custom-lora", "format_version": 1, "convention": "base(x) + x @ A @ B; scale=1", "modules": modules, "metadata": metadata, "state_dict": state, "resume_supported": False}
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)
    return {"path": str(path), "tensors": len(state), "bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}


def load_custom_lora(model, path, expected_base=None, expected_revision=None):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if payload.get("format") != "week09-custom-lora" or payload.get("format_version") != 1:
        raise ValueError("Expected explicit week09-custom-lora v1 checkpoint; legacy flat checkpoints must be migrated explicitly")
    if expected_base and payload["metadata"].get("base_model") != expected_base:
        raise ValueError("Checkpoint base model mismatch")
    if expected_revision and payload["metadata"].get("model_revision") != expected_revision:
        raise ValueError("Checkpoint base revision mismatch")
    current = custom_adapter_state(model)
    state = payload["state_dict"]
    if set(state) != set(current):
        raise ValueError("Adapter tensor names do not match; inject the same targets and rank first")
    parameters = dict(model.named_parameters())
    for name, value in state.items():
        if value.shape != current[name].shape or value.dtype != current[name].dtype or not torch.isfinite(value).all().item():
            raise ValueError(f"Invalid adapter tensor: {name}")
    with torch.no_grad():
        for name, value in state.items():
            parameters[name].copy_(value.to(parameters[name].device))
    return payload["metadata"]


def select_python_examples(stream, config):
    train, evaluation, seen = [], [], set()
    for scanned, example in enumerate(stream, 1):
        path = example.get("path", "")
        original = example.get("content", "")
        text = "\n".join(line[:config.max_characters] for line in original.split("\n"))
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        if path.endswith(".py") and text.strip() and digest not in seen:
            # Split by repository to reduce leakage between files from one project.
            repository = example.get("repo_name") or path
            bucket = int(hashlib.sha256(f"{config.seed}:{repository}".encode()).hexdigest(), 16) % 10
            destination, limit = (evaluation, config.eval_files) if bucket == 0 else (train, config.train_files)
            if len(destination) < limit:
                destination.append({"text": text, "sha256": digest, "path": path, "repo_name": repository, "original_characters": len(original), "processed_characters": len(text), "lines": len(original.split("\n"))})
                seen.add(digest)
        if len(train) == config.train_files and len(evaluation) == config.eval_files:
            break
        if scanned >= config.max_scanned:
            break
    if len(train) != config.train_files or len(evaluation) != config.eval_files:
        raise RuntimeError(f"Insufficient unique Python files within scan limit: train={len(train)}, eval={len(evaluation)}")
    assert {row["sha256"] for row in train}.isdisjoint(row["sha256"] for row in evaluation)
    assert {row["repo_name"] for row in train}.isdisjoint(row["repo_name"] for row in evaluation)
    return train, evaluation


def pack_examples(records, tokenizer, sequence_length):
    tokens = []
    for row in records:
        tokens.extend(tokenizer(row["text"], add_special_tokens=False)["input_ids"])
        tokens.append(tokenizer.eos_token_id)
    # Fixed 512-token blocks, packing multiple short files with EOS boundaries.
    blocks = [tokens[start:start + sequence_length] for start in range(0, len(tokens) - sequence_length + 1, sequence_length)]
    if not blocks:
        raise ValueError("Not enough tokens for one full sequence")
    return blocks


def collate_blocks(blocks, device):
    ids = torch.tensor(blocks, dtype=torch.long, device=device)
    return {"input_ids": ids, "attention_mask": torch.ones_like(ids)}


def causal_loss(logits, input_ids):
    # EOS is genuine data, never masked by matching its token ID to PAD.
    return F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]), input_ids[:, 1:].reshape(-1))


def autocast_for(device):
    return torch.autocast("cuda", dtype=torch.float16) if torch.device(device).type == "cuda" else nullcontext()


@torch.no_grad()
def evaluate_loss(model, blocks, device):
    was_training = model.training
    model.eval()
    total, tokens = 0.0, 0
    for block in blocks:
        batch = collate_blocks([block], device)
        with autocast_for(device):
            output = model(**batch, use_cache=False)
            loss = causal_loss(output.logits, batch["input_ids"])
        count = len(block) - 1
        total += loss.item() * count
        tokens += count
        del output, loss  # Do not retain vocabulary-sized logits for the next batch.
    model.train(was_training)
    nll = total / tokens
    assert math.isfinite(nll)
    return {"token_nll": nll, "perplexity": math.exp(nll) if nll < 700 else None, "predicted_tokens": tokens, "sequences": len(blocks)}


@torch.no_grad()
def generate_samples(model, tokenizer, device, prompts=PROMPTS, max_new_tokens=128):
    was_training = model.training
    model.eval()
    results = {}
    for prompt in prompts:
        if prompt:
            batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)
        else:
            start = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
            batch = collate_blocks([[start]], device)
        with autocast_for(device):
            output = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        results[prompt] = prompt + tokenizer.decode(output[0, batch["input_ids"].shape[1]:], skip_special_tokens=True)
    model.train(was_training)
    return results


def syntax_summary(samples):
    # Truncated continuations can fail ast.parse even if they resemble Python.
    parsed = 0
    for sample in samples.values():
        try:
            ast.parse(sample)
            parsed += bool(sample.strip())
        except (SyntaxError, ValueError):
            pass
    return {"parseable_nonempty_samples": parsed, "total_samples": len(samples), "limitation": "Syntax only; truncated Python and Python 2 can fail. No functional correctness claim."}


def gradient_smoke_test(model, block, device):
    """Backward only: leaves weights and optimizer state untouched."""
    model.train()
    model.zero_grad(set_to_none=True)
    try:
        batch = collate_blocks([block], device)
        with autocast_for(device):
            output = model(**batch, use_cache=False)
            loss = causal_loss(output.logits, batch["input_ids"])
        assert torch.isfinite(loss).item()
        loss.backward()
        adapters = [(name, module) for name, module in model.named_modules() if isinstance(module, CustomLoRALayer)]
        assert adapters
        for name, module in adapters:
            assert module.adapter_A.grad is not None and torch.isfinite(module.adapter_A.grad).all().item(), name
            assert module.adapter_B.grad is not None and torch.isfinite(module.adapter_B.grad).all().item(), name
            assert module.adapter_B.grad.norm().item() > 0, name
        assert all(parameter.grad is None for name, parameter in model.named_parameters() if not name.endswith((".adapter_A", ".adapter_B")))
        return {"loss": loss.item(), "modules_with_finite_nonzero_B_gradients": len(adapters)}
    finally:
        model.zero_grad(set_to_none=True)


def train_custom_lora(model, blocks, device, config, output_dir, metadata):
    """Fresh run only. Periodic files are inference weights, not resume checkpoints."""
    if getattr(model, "_week09_training_started", False):
        raise RuntimeError("Training already started on these weights; reload a fresh base for a new run")
    if config.max_steps < 1 or config.gradient_accumulation_steps < 1 or config.save_steps < 1:
        raise ValueError("Training steps, accumulation, and save interval must be positive")
    marker = Path(output_dir) / "TRAINING_STARTED.json"
    with marker.open("x") as file:
        json.dump({"config": asdict(config), "continuation": "unsupported: reload a fresh base and use a new output directory"}, file)
    model._week09_training_started = True
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    assert trainable and all(parameter.dtype == torch.float32 for parameter in trainable)
    optimizer = torch.optim.AdamW(trainable, lr=config.learning_rate)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.device(device).type == "cuda")
    generator = random.Random(config.seed)
    order = list(range(len(blocks)))
    generator.shuffle(order)
    cursor, history, completed, skipped = 0, [], 0, 0
    started = time.monotonic()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    while completed < config.max_steps:
        average_loss = 0.0
        for _ in range(config.gradient_accumulation_steps):
            if cursor == len(order):
                generator.shuffle(order)
                cursor = 0
            batch = collate_blocks([blocks[order[cursor]]], device)
            cursor += 1
            with autocast_for(device):
                output = model(**batch, use_cache=False)
                loss = causal_loss(output.logits, batch["input_ids"])
            if not torch.isfinite(loss).item():
                raise FloatingPointError("Nonfinite training loss")
            average_loss += loss.item() / config.gradient_accumulation_steps
            scaler.scale(loss / config.gradient_accumulation_steps).backward()
            del output, loss
        scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(trainable, 1.0)
        lr = config.learning_rate * min((completed + 1) / max(config.warmup_steps, 1), 1.0)
        for group in optimizer.param_groups:
            group["lr"] = lr
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        if scaler.get_scale() < previous_scale:
            skipped += 1
            if skipped > 20:
                raise FloatingPointError("Too many FP16 overflow steps; the run is incomplete")
            continue
        if not torch.isfinite(grad_norm).item():
            raise FloatingPointError("Nonfinite gradient norm")
        completed += 1
        entry = {"step": completed, "loss": average_loss, "learning_rate": lr, "grad_norm": float(grad_norm), "elapsed_seconds": time.monotonic() - started}
        history.append(entry)
        with (Path(output_dir) / "training_log.jsonl").open("a") as file:
            file.write(json.dumps(entry) + "\n")
        if completed == 1 or completed % 25 == 0:
            print(json.dumps(entry), flush=True)
        if completed % config.save_steps == 0:
            save_custom_lora(model, Path(output_dir) / f"adapter_step_{completed:04d}.pt", {**metadata, "global_step": completed, "config": asdict(config)})
    return {"global_step": completed, "skipped_overflow_steps": skipped, "mean_training_loss": sum(item["loss"] for item in history) / len(history), "elapsed_seconds": time.monotonic() - started, "history": history}


def configure_native_quantization(model_config):
    """Retain the checkpoint's JSON config for Transformers' pre-quantized path.

    Transformers 5.20.2 calls supports_quant_method(...).get() before it
    constructs BitsAndBytesConfig. Assigning that object here breaks dispatch.
    T4 requires an FP16 compute override; this does not requantize the weights.
    """
    native = getattr(model_config, "quantization_config", None)
    if not isinstance(native, dict):
        raise TypeError("Expected the pre-quantized checkpoint's native quantization dictionary")
    if native.get("quant_method") != "bitsandbytes" or not native.get("load_in_4bit", native.get("_load_in_4bit", False)):
        raise ValueError("Expected a native 4-bit bitsandbytes checkpoint")
    model_config.quantization_config = deepcopy(native)
    model_config.quantization_config["bnb_4bit_compute_dtype"] = "float16"
    return model_config


def load_cloud_base(config, revision):
    require_cloud_cuda()  # Must precede every from_pretrained call.
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
    from peft import prepare_model_for_kbit_training

    model_config = configure_native_quantization(AutoConfig.from_pretrained(config.base_model, revision=revision))
    tokenizer = AutoTokenizer.from_pretrained(config.base_model, revision=revision)
    tokenizer.pad_token = tokenizer.eos_token
    # No explicit quantization_config argument: the checkpoint is already NF4.
    model = AutoModelForCausalLM.from_pretrained(config.base_model, revision=revision, config=model_config, device_map={"": 0}, dtype=torch.float16, low_cpu_mem_usage=True, attn_implementation="sdpa")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    assert model.is_loaded_in_4bit
    assert all(parameter.device.type == "cuda" and parameter.device.index == 0 for parameter in model.parameters())
    quantized_modules = [module for module in model.modules() if hasattr(module, "compute_dtype")]
    assert quantized_modules and all(module.compute_dtype == torch.float16 for module in quantized_modules)
    return model, tokenizer

## Preserved earlier experiments

All old code is gated by `RUN_LEGACY_EXPERIMENTS`. Outputs remain exactly as recorded and do not indicate execution of the edited notebook. The optional old Trainer example disables automatic saving; the final task does not use Trainer or masquerade as a PEFT model. The original double-extension notebook remains untouched.

### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [1]:
if RUN_LEGACY_EXPERIMENTS:
    
    # Install missing packages using the final environment cell; no blanket upgrades.
    
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import transformers
    import peft
    import bitsandbytes
    
    from tqdm.auto import tqdm, trange
    
    assert torch.cuda.is_available(), "CUDA GPU required!"
    
    device = torch.device("cuda")
    
    print("=== Environment ===")
    print("PyTorch:", torch.__version__)
    print("Transformers:", transformers.__version__)
    print("PEFT:", peft.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    
    free, total = torch.cuda.mem_get_info()
    
    print(f"Total GPU memory: {total / 1024**3:.2f} GB")
    print(f"Free GPU memory: {free / 1024**3:.2f} GB")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
=== Environment ===
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
PEFT: 0.21.3
GPU: Tesla T4
Total GPU memory: 14.56 GB
Free GPU memory: 14.46 GB


In [2]:
if RUN_LEGACY_EXPERIMENTS:
    model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token_id = tokenizer.eos_token_id
    
    # the main model weights are loaded in 4-bit precision - but we can still tune LoRAs later
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
        quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
    )
    for param in model.parameters():
        param.requires_grad=False
    model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
    model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
    # more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174


config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.07GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [5]:
if RUN_LEGACY_EXPERIMENTS:
    prompt = 'A quick brown fox'
    batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    
    for i in range(10):
        next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
        batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
        batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)
    
    print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))



Output: A quick brown fox jumps over the lazy dog. The quick brown fox


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [23]:
if RUN_LEGACY_EXPERIMENTS:
    the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
    batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
    outputs = model(**batch)
    
    next_word_logits = outputs.logits[:, :-1]
    true_next_tokens = batch['input_ids'][:, 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    
    print("Loss:", loss)


Loss: tensor(3.5394, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [6]:
if RUN_LEGACY_EXPERIMENTS:
    class WordEmbeddingsWithLearnedPrompts(nn.Module):
        """
        To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
         - that inserts trainable prompts instead of the first N token embeddings. """
    
        def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
            super().__init__()
            self.original_word_embeddings = word_embeddings
            self.num_prompts = num_prompts
            self.learnable_prompts = nn.Parameter(
                torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)
    
        def forward(self, input_ids: torch.LongTensor):
            # input_ids shape: [batch_size, seq length]
            assert input_ids.dtype == torch.int64
            assert input_ids.shape[1] > self.num_prompts
            assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"
    
            # Your task: embed input_ids, but replace the first :num_prompts: tokens with self.learnable_prompts
            # This is because we will prepend :num_prompts: padding tokens at the beginning
    
            # After you are done, you must produce a word embedding vector for each token in input_ids,
            # except that the first :num_prompts: vectors should equal learnable_prompts;
            # any additional vectors after first :num_prompts: ones should be embedded as usual
            # Note: since you're dealing with trainable params, please torch.cat instead of item assignment
    
            
            token_embeddings = self.original_word_embeddings(input_ids[:, self.num_prompts:])
            prompt_embeddings = self.learnable_prompts.expand(input_ids.shape[0], -1, -1).to(dtype=token_embeddings.dtype)
            outputs = torch.cat([prompt_embeddings, token_embeddings], dim=1)
            return outputs
        


In [7]:
if RUN_LEGACY_EXPERIMENTS:
    num_prompts = 16
    test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
    test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)
    
    space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                                   dtype=torch.int64, device=device)
    test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)
    
    with torch.cuda.amp.autocast():
      test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)
    
    assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
    assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
    assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
    assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
    print("Looks legit!")


/tmp/ipykernel_50/1152930240.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Looks legit!


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [8]:
if RUN_LEGACY_EXPERIMENTS:
    assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"
    
    model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
    
    opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)


In [13]:
if RUN_LEGACY_EXPERIMENTS:
    the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
    batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
    space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                                   dtype=torch.int64, device=device)
    batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
    batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)
    
    opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)
    model.train()
    
    loss_history = []
    
    for step in range(1, 301):
        opt.zero_grad(set_to_none=True)
        outputs = model(**batch, use_cache=False)
        logits = outputs.logits[:, num_prompts:-1, :]
        targets = batch["input_ids"][:, num_prompts+1:]
        loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), targets.reshape(-1))
        loss.backward()
        opt.step()
        loss_history.append(loss.item())
    
        if step == 1 or step % 20 == 0 or loss.item() <= 0.1:
            print(f"Step {step:3d} | Loss: {loss.item():.4f}")
    
        if loss.item() <= 0.1:
            print("Target reached!")
            break
    
    
    assert loss.item() <= 0.1
    print("Good job!")


Step   1 | Loss: 4.1343
Step  20 | Loss: 1.8332
Step  31 | Loss: 0.0567
Target reached!
Good job!


In [15]:
if RUN_LEGACY_EXPERIMENTS:
    prompt = 'A quick brown fox'
    batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
    batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)
    
    
    for i in range(15):
        next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
        batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
        batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)
    
    print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))
    
    # if you did everything right, the model will deny that the fox jumped over the lazy dog



Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


### Using HuggingFace PEFT

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [20]:
if RUN_LEGACY_EXPERIMENTS:
    import peft
    if isinstance(model.model.embed_tokens, WordEmbeddingsWithLearnedPrompts):
        model.model.embed_tokens = model.model.embed_tokens.original_word_embeddings
    assert isinstance(model.model.embed_tokens, nn.Embedding)
    
    peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
    model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
    print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
    print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))


Trainable parameters: 65536
Total parameters (excluding quantization): 4717917184


In [28]:
if RUN_LEGACY_EXPERIMENTS:
    # Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
    # Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
    # Finally, generate the sentence to make sure that the model learned the truth.
    
    # Train Hugging Face PEFT soft prompts
    the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
    batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)
    
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.01)
    model.train()
    
    for step in range(1, 201):
        opt.zero_grad(set_to_none=True)
        outputs = model(**batch, use_cache=False)
        logits = outputs.logits[:, 16:-1, :]
        targets = batch["input_ids"][:, 1:]
        loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), targets.reshape(-1))
        loss.backward()
        opt.step()
    
        if step == 1 or step % 20 == 0 or loss.item() <= 0.1:
            print(f"Step {step} | Loss: {loss.item():.4f}")
    
        if loss.item() <= 0.1:
            print("Target reached!")
            break
    
    model.eval()
    print("Final Loss:", loss.item())


Step 1 | Loss: 3.7378
Step 20 | Loss: 1.3523
Step 29 | Loss: 0.0857
Target reached!
Final Loss: 0.08567328006029129


In [32]:
if RUN_LEGACY_EXPERIMENTS:
    # Feel free to structure your 
    
    inputs = tokenizer("A quick brown fox", return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    
    print(tokenizer.decode(generated[0], skip_special_tokens=True))


[transformers] Both `max_new_tokens` (=20) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway! Besides, have you


### Parameter-efficient finetuning with LoRA (1 point)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [3]:
if RUN_LEGACY_EXPERIMENTS:
    import peft  # v-- re-load the notebook to remove any previous PEFT tuners, run cell 1 and this
    model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model = transformers.AutoModelForCausalLM.from_pretrained(
        model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
        quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
    ).to(device)
    model = peft.prepare_model_for_kbit_training(model, use_gradient_checkpointing=True) # < checkpoints and input grads


config.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

You shouldn't move a model that is dispatched using accelerate hooks.


In [6]:
if RUN_LEGACY_EXPERIMENTS:
    class LoRALayer(nn.Module):
        """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
        def __init__(self, module: nn.Linear, rank: int):
            super().__init__()
            self.module = module  # pre-trained (frozen) linear layer
            self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
            nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
            self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))
    
        def forward(self, input):
       
            base_output = self.module(input)
            lora_output = (input.to(self.adapter_A.dtype) @ self.adapter_A @ self.adapter_B).to(base_output.dtype)
            return base_output + lora_output


In [5]:
if RUN_LEGACY_EXPERIMENTS:
    # test your implementation
    test_linear = nn.Linear(128, 128)
    test_linear.weight.data[...] = torch.eye(128)
    test_adapter = LoRALayer(test_linear, rank=8)
    
    assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"
    
    test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
    test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
    test_linear.bias.data[...] = torch.linspace(1., -1., 128)
    
    dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
    assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
    dummy_loss.backward()
    assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
    assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
    assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
    # note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
    del dummy_loss, test_linear, test_adapter
    print("All tests passed!")


All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in attention blocks. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [7]:
if RUN_LEGACY_EXPERIMENTS:
    
    lora_rank = 8
    
    for layer in model.model.layers:
        for name in ("q_proj", "k_proj", "v_proj"):
            proj = getattr(layer.self_attn, name)
            if isinstance(proj, LoRALayer):
                continue
            if hasattr(proj, "adapter_A") and hasattr(proj, "module"):
                proj = proj.module
            assert hasattr(proj, "in_features") and hasattr(proj, "out_features"), f"Unsupported layer: {type(proj).__name__}"
            setattr(layer.self_attn, name, LoRALayer(proj, rank=lora_rank))
    
    n_lora = sum(isinstance(m, LoRALayer) for m in model.modules())
    expected = 3 * len(model.model.layers)
    
    print("LoRA layers:", n_lora)
    print("Expected:", expected)
    assert n_lora == expected
    print("LoRA injection successful!")


LoRA layers: 108
Expected: 108
LoRA injection successful!


In [8]:
if RUN_LEGACY_EXPERIMENTS:
    
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Trainable percentage: {100 * trainable / total:.4f}%")
    print("LoRA modules:", sum(isinstance(m, LoRALayer) for m in model.modules()))


Total parameters: 4,723,160,064
Trainable parameters: 5,308,416
Trainable percentage: 0.1124%
LoRA modules: 108


In [9]:
if RUN_LEGACY_EXPERIMENTS:
    
    
    # Patch LoRA forward without rebuilding the model
    def lora_forward_fixed(self, x):
        base_output = self.module(x)
        lora_input = x.to(device=self.adapter_A.device, dtype=self.adapter_A.dtype)
        lora_output = (lora_input @ self.adapter_A @ self.adapter_B).to(device=base_output.device, dtype=base_output.dtype)
        return base_output + lora_output
    
    LoRALayer.forward = lora_forward_fixed
    
    print("LoRA modules:", sum(isinstance(m, LoRALayer) for m in model.modules()))
    print("LoRA forward patched!")
    


LoRA modules: 108
LoRA forward patched!


In [10]:
if RUN_LEGACY_EXPERIMENTS:
    batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
    # test a single training step, make sure we get meaningful gradients
    with torch.cuda.amp.autocast(dtype=torch.float32):
        out = model.forward(**batch)
        (out.logits.norm() / 100).backward()
    
    for i, module in enumerate(model.modules()):
        if isinstance(module, LoRALayer):
            assert module.adapter_B.grad is not None
            assert module.adapter_B.grad.norm().item() > 0
    
    model.zero_grad(set_to_none=True)
    print("Grad check successful, well done!")


/tmp/ipykernel_51/2281301266.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [3]:
if RUN_LEGACY_EXPERIMENTS:
    # checking if the model can learn. Change max_steps for proper training
    import datasets
    data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
    data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
    model._hf_peft_config_loaded = True  # Legacy Trainer quantization gate only; custom final loop needs no PEFT flag.
    
    trainer = transformers.Trainer(
        model=model, train_dataset=data,
        args=transformers.TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=1,
            # note: if you want larger batch size, increase gradient_accumulation_steps
            warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
            save_strategy="no", logging_steps=1, output_dir='outputs', report_to="none"),
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
    )
    # if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!
    
    
    
    
    trainer.train()
    
    # NOTE: this is just an example! you do not have to wait for this progressbar to finish :)


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [13]:
if RUN_LEGACY_EXPERIMENTS:
    # Explicit custom checkpoint format; never use save_pretrained for this adapter.
    assert trainer.state.global_step == 100, "Save the already completed example; do not retrain it."
    save_path = "/kaggle/working/week09_example_custom_lora_v1.pt"
    assert len(custom_adapter_state(model)) == 216
    checkpoint = save_custom_lora(model, save_path, {"base_model": model_name, "rank": 8, "global_step": 100, "experiment": "historical_quotes_example"})
    print(checkpoint)


Completed steps: 100
Saved tensors: 216
File size: 20.32 MB
LoRA checkpoint saved: /kaggle/working/week09_lora_trained.pt


### Final task: *actually* train the model (3 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

## Final assignment — smoke first, then a fresh full experiment

Run with SMOKE_ONLY=True first (two real GPU optimizer updates, evaluation and reload). In a fresh kernel, set SMOKE_ONLY=False and SMOKE_RESULTS_PATH to the successful results.json for a fresh 500-update run. The smoke gate checks code, source revisions, GPU, environment and hyperparameters. Reusing an output directory is rejected.

Use the canonical codeparrot/codeparrot-clean repository (its only split is train). Its README links a smaller validation repository, but this notebook retains the exact dataset identity requested. Stream records, truncate **every line** to 512 characters, split by repository hash, deduplicate processed content, and pack EOS-separated files into exact 512-token blocks. Source text is not globally cropped to 512 characters.

Custom adapter checkpoints contain weights and metadata for inference; exact optimizer resume is unsupported. See CLOUD_EXECUTION_REPORT.md for commands and retrieval instructions.


In [ ]:
if RUN_FINAL_EXPERIMENT:
    require_cloud_cuda()
    import importlib.util
    import subprocess
    import sys
    # Install missing dependencies only; retain the Kaggle image's existing versions.
    requirements = {"transformers": "transformers>=4.56,<6", "peft": "peft>=0.15,<1", "bitsandbytes": "bitsandbytes>=0.45.5,<1", "accelerate": "accelerate>=1,<2", "datasets": "datasets>=3,<6", "huggingface_hub": "huggingface_hub>=0.30,<2"}
    missing = [requirement for module, requirement in requirements.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    import importlib.metadata
    from packaging.version import Version
    assert Version(importlib.metadata.version("transformers")) >= Version("4.56"), "The dtype= loading API used here needs Transformers >=4.56. Install a compatible version explicitly and restart."
    assert Version(torch.__version__.split("+")[0]) >= Version("2.3"), "Use a recent Kaggle PyTorch image."
    print({package: importlib.metadata.version(package) for package in ["torch", *requirements]})

In [ ]:
if RUN_FINAL_EXPERIMENT:
    from huggingface_hub import HfApi
    import datasets
    config = ExperimentConfig()
    if SMOKE_ONLY:
        config.max_steps = 2
        config.save_steps = 2
        config.train_files = 32
        config.eval_files = 16
        config.max_new_tokens = 16
    seed_everything(config.seed)
    run_name = EXPERIMENT_NAME + ("_smoke" if SMOKE_ONLY else "")
    output_dir = Path.cwd() / run_name
    import shutil
    assert shutil.disk_usage(Path.cwd()).free >= 15 * 2**30, "Need at least 15 GiB free checkpoint/cache storage"
    torch.cuda.set_device(0)
    torch.cuda.reset_peak_memory_stats(0)
    memory_before = memory_snapshot("before_model_load")
    assert memory_before["devices"][0]["free_gib"] >= 13, "Free CUDA 0 by restarting the Kaggle session."
    api = HfApi()
    # Resolve immutable source revisions before downloading any model weights.
    model_revision = api.model_info(config.base_model).sha
    dataset_revision = api.dataset_info(config.dataset).sha
    environment = {package: importlib.metadata.version(package) for package in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "datasets", "huggingface_hub"]}
    environment.update({"python": sys.version.split()[0], "cuda": str(torch.version.cuda)})
    metadata = {"base_model": config.base_model, "model_revision": model_revision, "dataset": config.dataset, "dataset_revision": dataset_revision, "dataset_split": "train", "seed": config.seed, "environment": environment, "smoke_only": SMOKE_ONLY, "gpu_strategy": "single_process_cuda_0", "execution_device": "cuda:0", "gpu_name": torch.cuda.get_device_name(0), "helper_sha256": HELPER_SOURCE_SHA256}
    if not SMOKE_ONLY:
        metadata["smoke_gate"] = validate_smoke_gate(SMOKE_RESULTS_PATH, config, metadata)
    output_dir.mkdir(exist_ok=False)  # Protect previous artifacts and accidental reruns.
    (output_dir / "run_config.json").write_text(json.dumps({"config": asdict(config), "metadata": metadata}, indent=2))
    metadata["dataset_split"] = "train"  # Canonical repository exposes train only; split by repository below.
    stream = datasets.load_dataset(config.dataset, split="train", revision=dataset_revision, streaming=True)
    train_records, eval_records = select_python_examples(stream, config)
    model, tokenizer = load_cloud_base(config, model_revision)
    model_load_passed = True  # Only reached after real NF4 load, FP16 and CUDA placement assertions.
    print("MODEL_LOAD_PASSED: native NF4 checkpoint on CUDA 0, FP16 compute", flush=True)
    device = torch.device("cuda:0")
    train_blocks = pack_examples(train_records, tokenizer, config.sequence_length)
    eval_blocks = pack_examples(eval_records, tokenizer, config.sequence_length)
    assert all(len(block) == 512 for block in train_blocks + eval_blocks)
    manifest = {"train": [{key: value for key, value in row.items() if key != "text"} for row in train_records], "eval": [{key: value for key, value in row.items() if key != "text"} for row in eval_records]}
    manifest_path = output_dir / "data_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))
    metadata["data_manifest_sha256"] = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    torch.save({"train": train_blocks, "eval": eval_blocks}, output_dir / "token_blocks.pt")
    tokenizer.save_pretrained(output_dir / "tokenizer")
    print({"train_files": len(train_records), "eval_files": len(eval_records), "train_sequences": len(train_blocks), "eval_sequences": len(eval_blocks), "sequence_length": config.sequence_length})
    memory_loaded = memory_snapshot("base_model_loaded")

In [ ]:
if RUN_FINAL_EXPERIMENT:
    # Capture the untouched base on exactly the same held-out sequences and prompts.
    baseline_metrics = evaluate_loss(model, eval_blocks, device)
    before_samples = generate_samples(model, tokenizer, device, max_new_tokens=config.max_new_tokens)
    (output_dir / "baseline.json").write_text(json.dumps({"metrics": baseline_metrics, "syntax": syntax_summary(before_samples), "samples": before_samples}, indent=2))
    base_probe_metrics = evaluate_loss(model, eval_blocks[:2], device)
    lora_modules = inject_custom_lora(model, config.rank)
    assert len(lora_modules) == 216  # 36 decoder layers x (3 attention + 3 MLP).
    assert len(custom_adapter_state(model)) == 432
    zero_adapter_metrics = evaluate_loss(model, eval_blocks[:2], device)
    assert abs(zero_adapter_metrics["token_nll"] - base_probe_metrics["token_nll"]) < 1e-5, "Zero-initialized LoRA changed base loss"
    probe_batch = collate_blocks([eval_blocks[0]], device)
    assert all(torch.count_nonzero(module.adapter_B).item() == 0 for _, module in lora_modules)
    smoke_metrics = gradient_smoke_test(model, train_blocks[0], device)
    print({"baseline": baseline_metrics, "gradient_smoke": smoke_metrics, "lora_modules": len(lora_modules), "trainable_parameters": sum(value.numel() for value in model.parameters() if value.requires_grad)})
    memory_smoke = memory_snapshot("after_full_512_token_backward_smoke")

In [ ]:
if RUN_FINAL_EXPERIMENT:
    # 500 successful optimizer updates; micro-batch 1 x accumulation 4 = batch 4.
    # No Trainer, automatic PEFT checkpointing, or implicit continuation.
    if not SMOKE_ONLY:
        metadata["smoke_gate"] = validate_smoke_gate(SMOKE_RESULTS_PATH, config, metadata)
    initial_adapter_state = custom_adapter_state(model)
    training_metrics = train_custom_lora(model, train_blocks, device, config, output_dir, metadata)
    assert training_metrics["global_step"] == config.max_steps
    changed_adapter_tensors = sum(not torch.equal(value, initial_adapter_state[name]) for name, value in custom_adapter_state(model).items())
    assert changed_adapter_tensors > 0, "Optimizer did not change adapter weights"
    del initial_adapter_state
    checkpoint_info = save_custom_lora(model, output_dir / "adapter_final.pt", {**metadata, "global_step": training_metrics["global_step"], "config": asdict(config)})
    print(checkpoint_info)

In [ ]:
if RUN_FINAL_EXPERIMENT:
    after_metrics = evaluate_loss(model, eval_blocks, device)
    after_samples = generate_samples(model, tokenizer, device, max_new_tokens=config.max_new_tokens)
    # Destructive-in-memory reload smoke: zero adapters, restore from disk, compare.
    model.eval()
    with torch.no_grad(), autocast_for(device):
        logits_before_reload = model(**probe_batch, use_cache=False).logits[:, -1].float().cpu()
    with torch.no_grad():
        for _, module in lora_modules:
            module.adapter_A.zero_()
            module.adapter_B.zero_()
    restored_metadata = load_custom_lora(model, output_dir / "adapter_final.pt", config.base_model, model_revision)
    with torch.no_grad(), autocast_for(device):
        logits_after_reload = model(**probe_batch, use_cache=False).logits[:, -1].float().cpu()
    assert torch.allclose(logits_before_reload, logits_after_reload, atol=1e-4, rtol=1e-4), "Checkpoint reload changed logits"
    assert any(torch.count_nonzero(module.adapter_B).item() for _, module in lora_modules)
    reload_samples = generate_samples(model, tokenizer, device, max_new_tokens=config.max_new_tokens)
    assert reload_samples == after_samples, "Reloaded greedy generations must match"
    memory_final = memory_snapshot("after_training_and_reload")
    results = {"status": "smoke_only" if SMOKE_ONLY else "completed", "metadata": metadata, "config": asdict(config), "model_load_passed": model_load_passed, "gpu_smoke_passed": bool(SMOKE_ONLY and training_metrics["global_step"] == 2), "baseline": baseline_metrics, "after": after_metrics, "nll_change": after_metrics["token_nll"] - baseline_metrics["token_nll"], "syntax_before": syntax_summary(before_samples), "syntax_after": syntax_summary(after_samples), "training": {key: value for key, value in training_metrics.items() if key != "history"}, "gradient_smoke": smoke_metrics, "changed_adapter_tensors": changed_adapter_tensors, "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad), "checkpoint_reload_logits_passed": True, "checkpoint_reload_generations_passed": True, "checkpoint": checkpoint_info, "before_samples": before_samples, "after_samples": after_samples, "memory": [memory_before, memory_loaded, memory_smoke, memory_final]}
    if SMOKE_ONLY:
        validate_path = output_dir / "smoke_candidate.json"
        validate_path.write_text(json.dumps(results, indent=2))
        validate_smoke_gate(validate_path, config, metadata)
        validate_path.unlink()
        print("GPU_SMOKE_PASSED: two updates and adapter reload checks completed", flush=True)
    (output_dir / "results.json").write_text(json.dumps(results, indent=2))
    print(json.dumps({key: results[key] for key in ["status", "baseline", "after", "nll_change", "training", "checkpoint_reload_logits_passed"]}, indent=2))

In [ ]:
if RUN_FINAL_EXPERIMENT:
    from html import escape
    from IPython.display import HTML, display
    rows = ["<tr><th>Prompt</th><th>Before</th><th>After</th></tr>"]
    for prompt in PROMPTS:
        rows.append(f"<tr><td><pre>{escape(prompt or '(empty)')}</pre></td><td><pre>{escape(before_samples[prompt])}</pre></td><td><pre>{escape(after_samples[prompt])}</pre></td></tr>")
    comparison_html = '<table border="1" style="table-layout:fixed;width:100%;white-space:pre-wrap">' + "".join(rows) + "</table>"
    (output_dir / "comparison.html").write_text(comparison_html)
    display(HTML(comparison_html))
    improvement = baseline_metrics["token_nll"] - after_metrics["token_nll"]
    report = f"""# Week 09 PEFT: Python code generation

Run status: {results['status']}. Model: {config.base_model}, revision {model_revision}.
Dataset: {config.dataset}, revision {dataset_revision}. Repository-disjoint subsets: {len(train_records)} training and {len(eval_records)} evaluation files. Each line was cropped to {config.max_characters} characters and packed into {config.sequence_length}-token sequences.

Our custom, unscaled rank-{config.rank} LoRA adapters were applied to Q/K/V attention projections and gate/up/down MLP projections ({len(lora_modules)} modules). Only adapters were optimized, using FP16 autocast with FP32 adapters, a frozen NF4 backbone, micro-batch 1, and {config.gradient_accumulation_steps} accumulation steps. Completed optimizer updates: {training_metrics['global_step']}.

Held-out token NLL: {baseline_metrics['token_nll']:.6f} before, {after_metrics['token_nll']:.6f} after. Perplexity: {baseline_metrics['perplexity']:.4f} before, {after_metrics['perplexity']:.4f} after. NLL improvement (positive is better): {improvement:.6f}. Training mean loss: {training_metrics['mean_training_loss']:.6f}. Runtime: {training_metrics['elapsed_seconds']:.1f} seconds.

Greedy continuations for eight fixed prompts are in comparison.html. Parseable nonempty samples: {syntax_summary(before_samples)['parseable_nonempty_samples']}/8 before and {syntax_summary(after_samples)['parseable_nonempty_samples']}/8 after. Syntax parsing is only descriptive: truncated continuations or Python 2 code may fail, and no functional correctness is asserted.

Adapter-only saving/loading passed matching-logit and identical-generation checks. Full numeric details, GPU memory measurements, environment versions, and immutable source revisions are in results.json. This small experiment does not establish generalization beyond the selected held-out subset; numerical determinism of quantized CUDA kernels is not guaranteed.
"""
    (output_dir / "coursework_report.md").write_text(report)
    print(report)
    print("Download all files from", output_dir, "using Kaggle Save Version → Save & Run All → Output.")

## Completion checks

A full assignment run requires `SMOKE_ONLY = False`, `results.json` status `completed`, `training.global_step = 500`, 216 LoRA modules / 432 adapter tensors, finite held-out losses before and after, gradient coverage, and both checkpoint reload checks. Improvement is reported from measured values, never assumed. Download `adapter_final.pt`, tokenizer files, run config, token blocks, file manifest, logs, metrics, comparison HTML, report, and the executed notebook. A weights-only checkpoint cannot exactly resume optimization.

References: [CodeParrot dataset](https://huggingface.co/datasets/codeparrot/codeparrot-clean), [smaller validation repository](https://huggingface.co/datasets/codeparrot/codeparrot-clean-valid), [model configuration](https://huggingface.co/unsloth/Qwen3-8B-Base-bnb-4bit/blob/main/config.json), [Hugging Face bitsandbytes documentation](https://huggingface.co/docs/transformers/quantization/bitsandbytes), [Kaggle notebooks](https://www.kaggle.com/docs/notebooks).

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.